### Q1. Generating questions

In [1]:
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


load_dotenv()
openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print(len(documents))

72


In [3]:
from evaluation_utils import llm_structured

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [6]:
records = []
input_tokens = []

for doc in documents[:3]:

    user_prompt = json.dumps(
        {
            "filename": doc["filename"],
            "content": doc["content"]
        },
        indent=2,
    )

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )

    input_tokens.append(usage.input_tokens)

    for q in result.questions:
        records.append(
            {
                "question": q,
                "filename": doc["filename"]
            }
        )

print(input_tokens)
print("Average:", sum(input_tokens)/len(input_tokens))

[1024, 1290, 1757]
Average: 1357.0


### Q2. First result with text search

In [7]:
ground_truth = (
    pd.read_csv("ground-truth.csv")
      .to_dict(orient="records")
)

len(ground_truth)

360

In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

In [9]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [10]:
def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        num_results=num_results
    )

In [11]:
from minsearch import VectorSearch
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

vector_index = VectorSearch(keyword_fields=["filename"])
texts = [c["content"] for c in chunks]
vectors = embedding_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
vector_index.fit(vectors=vectors, payload=chunks)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [18]:
def vector_search(query, num_results=5):
    query_vector = embedding_model.encode(
        query,
        convert_to_numpy=True
    )

    return vector_index.search(
        query_vector,
        num_results=num_results
    )

In [13]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [14]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

### Q2. First result with text search

In [15]:
q = ground_truth[0]["question"]

result = text_search(q)

print(result[0]["filename"])

01-agentic-rag/lessons/03-rag.md


### Q3. First result with vector search

In [19]:
q = ground_truth[0]["question"]

vector_results = vector_search(q)

print(len(vector_results))

print(vector_results[0]["filename"])

5
01-agentic-rag/lessons/01-intro.md


### Evaluation functions

In [20]:
def compute_relevance(results, filename):

    return [
        int(doc["filename"] == filename)
        for doc in results
    ]

In [21]:
def hit_rate(relevance_total):

    cnt = 0

    for rel in relevance_total:

        if 1 in rel:
            cnt += 1

    return cnt / len(relevance_total)

In [22]:
def mrr(relevance_total):

    total = 0

    for rel in relevance_total:

        score = 0

        for rank, value in enumerate(rel):

            if value == 1:
                score = 1 / (rank + 1)
                break

        total += score

    return total / len(relevance_total)

In [23]:
def evaluate(ground_truth, search_function):

    relevance_total = []

    for row in ground_truth:

        results = search_function(row["question"])

        relevance = compute_relevance(
            results,
            row["filename"]
        )

        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

### Q4. Evaluating text search

In [24]:
evaluate(
    ground_truth,
    text_search
)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

### Q5. Evaluating vector search

In [25]:
evaluate(
    ground_truth,
    vector_search
)

{'hit_rate': 0.8333333333333334, 'mrr': 0.6734722222222222}

### Q6. Tuning hybrid search

In [26]:
for k in [1, 50, 100, 200]:

    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )

    print(k, result)

1 {'hit_rate': 0.8555555555555555, 'mrr': 0.6995833333333333}
50 {'hit_rate': 0.85, 'mrr': 0.6760648148148148}
100 {'hit_rate': 0.85, 'mrr': 0.6760648148148148}
200 {'hit_rate': 0.85, 'mrr': 0.6760648148148148}
